In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import pathlib
import random
import string
import re
import numpy as np

import tensorflow.data as tf_data
import tensorflow.strings as tf_strings

import keras
from keras import layers
from keras import ops
from keras.layers import TextVectorization

In [ ]:
text_file = "spa.txt"

In [ ]:
with open(text_file) as f:
    lines = f.read().split("\n")[:-1]

text_pairs = []
for line in lines:
    eng, spa, _ = line.split("\t")
    spa = "[start] " + spa + " [end]"
    text_pairs.append((eng, spa))

In [ ]:
for _ in range(5):
    print(random.choice(text_pairs))

('He erased his speech from the tape.', '[start] Él borró su discurso de la cinta. [end]')
('In the summer, people go to the seaside.', '[start] En verano, la gente va a la playa. [end]')
('She gave a big pull on the rope.', '[start] Le pegó un buen tirón a la cuerda. [end]')
("I don't speak your language.", '[start] No hablo tu idioma. [end]')
("He doesn't get jokes.", '[start] Él no entiende las bromas. [end]')


In [ ]:
random.shuffle(text_pairs)
num_val_samples = int(0.15 * len(text_pairs))
num_train_samples = len(text_pairs) - 2 * num_val_samples
train_pairs = text_pairs[:num_train_samples]
val_pairs = text_pairs[num_train_samples : num_train_samples + num_val_samples]
test_pairs = text_pairs[num_train_samples + num_val_samples :]

print(f"{len(text_pairs)} total pairs")
print(f"{len(train_pairs)} training pairs")
print(f"{len(val_pairs)} validation pairs")
print(f"{len(test_pairs)} test pairs")

141543 total pairs
99081 training pairs
21231 validation pairs
21231 test pairs


In [ ]:
strip_chars = string.punctuation + "¿"
strip_chars = strip_chars.replace("[", "")
strip_chars = strip_chars.replace("]", "")

vocab_size = 15000
sequence_length = 20
batch_size = 128

In [ ]:
def custom_standardization(input_string):
    lowercase = tf_strings.lower(input_string)
    return tf_strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")


eng_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

spa_vectorization = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length + 1,
    standardize=custom_standardization,
)

train_eng_texts = [pair[0] for pair in train_pairs]
train_spa_texts = [pair[1] for pair in train_pairs]
eng_vectorization.adapt(train_eng_texts)
spa_vectorization.adapt(train_spa_texts)

In [ ]:
def format_dataset(eng, spa):
    eng = eng_vectorization(eng)
    spa = spa_vectorization(spa)
    return (
        {
            "encoder_inputs": eng,
            "decoder_inputs": spa[:, :-1],
        },
        spa[:, 1:],
    )


def make_dataset(pairs):
    eng_texts, spa_texts = zip(*pairs)
    eng_texts = list(eng_texts)
    spa_texts = list(spa_texts)
    dataset = tf_data.Dataset.from_tensor_slices((eng_texts, spa_texts))
    dataset = dataset.batch(batch_size)
    dataset = dataset.map(format_dataset)
    return dataset.cache().shuffle(2048).prefetch(16)


train_ds = make_dataset(train_pairs)
val_ds = make_dataset(val_pairs)

In [ ]:
for inputs, targets in train_ds.take(1):
    print(f'inputs["encoder_inputs"].shape: {inputs["encoder_inputs"].shape}')
    print(f'inputs["decoder_inputs"].shape: {inputs["decoder_inputs"].shape}')
    print(f"targets.shape: {targets.shape}")

inputs["encoder_inputs"].shape: (128, 20)
inputs["decoder_inputs"].shape: (128, 20)
targets.shape: (128, 20)


In [ ]:
import keras.ops as ops


class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
        else:
            padding_mask = None

        attention_output = self.attention(
            query=inputs, value=inputs, key=inputs, attention_mask=padding_mask
        )
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim
        )
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim

    def call(self, inputs):
        length = ops.shape(inputs)[-1]
        positions = ops.arange(0, length, 1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def compute_mask(self, inputs, mask=None):
        if mask is None:
            return None
        else:
            return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


In [ ]:
class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.latent_dim = latent_dim
        self.num_heads = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(latent_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask = ops.cast(mask[:, None, :], dtype="int32")
            padding_mask = ops.minimum(padding_mask, causal_mask)
        else:
            padding_mask = None

        attention_output_1 = self.attention_1(
            query=inputs, value=inputs, key=inputs, attention_mask=causal_mask
        )
        out_1 = self.layernorm_1(inputs + attention_output_1)

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=padding_mask,
        )
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        return self.layernorm_3(out_2 + proj_output)

    def get_causal_attention_mask(self, inputs):
        input_shape = ops.shape(inputs)
        batch_size, sequence_length = input_shape[0], input_shape[1]
        i = ops.arange(sequence_length)[:, None]
        j = ops.arange(sequence_length)
        mask = ops.cast(i >= j, dtype="int32")
        mask = ops.reshape(mask, (1, input_shape[1], input_shape[1]))
        mult = ops.concatenate(
            [ops.expand_dims(batch_size, -1), ops.convert_to_tensor([1, 1])],
            axis=0,
        )
        return ops.tile(mask, mult)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "latent_dim": self.latent_dim,
                "num_heads": self.num_heads,
            }
        )
        return config

In [ ]:
embed_dim = 256
latent_dim = 2048
num_heads = 8

encoder_inputs = keras.Input(shape=(None,), dtype="int64", name="encoder_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(encoder_inputs)
encoder_outputs = TransformerEncoder(embed_dim, latent_dim, num_heads)(x)
encoder = keras.Model(encoder_inputs, encoder_outputs)

decoder_inputs = keras.Input(shape=(None,), dtype="int64", name="decoder_inputs")
encoded_seq_inputs = keras.Input(shape=(None, embed_dim), name="decoder_state_inputs")
x = PositionalEmbedding(sequence_length, vocab_size, embed_dim)(decoder_inputs)
x = TransformerDecoder(embed_dim, latent_dim, num_heads)(x, encoded_seq_inputs)
x = layers.Dropout(0.5)(x)
decoder_outputs = layers.Dense(vocab_size, activation="softmax")(x)
decoder = keras.Model([decoder_inputs, encoded_seq_inputs], decoder_outputs)

decoder_outputs = decoder([decoder_inputs, encoder_outputs])
transformer = keras.Model(
    [encoder_inputs, decoder_inputs], decoder_outputs, name="transformer"
)

In [ ]:
epochs = 10

transformer.summary()
transformer.compile(
    "rmsprop", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)

Model: "transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs            │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ positional_embedding      │ (None, None, 256)      │      3,845,120 │ encoder_inputs[0][0]   │
│ (PositionalEmbedding)     │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ decoder_inputs            │ (None, None)           │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ transformer_encoder       │ (None, None, 256)      │      3,155,456 │ positional_embedding[… │
│ (TransformerEncoder)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ functional_3 (Functional) │ (None, None, 15000)    │     12,959,640 │ decoder_inputs[0][0],  │
│                           │                        │                │ transformer_encoder[0… │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 39,920,434 (152.28 MB)

 Trainable params: 19,960,216 (76.14 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 19,960,218 (76.14 MB)

Epoch 1/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 102s 116ms/step - accuracy: 0.7282 - loss: 1.7819 - val_accuracy: 0.7787 - val_loss: 1.3914
Epoch 2/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 122s 97ms/step - accuracy: 0.7678 - loss: 1.4542 - val_accuracy: 0.9291 - val_loss: 0.5316
Epoch 3/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 82s 98ms/step - accuracy: 0.9033 - loss: 0.6971 - val_accuracy: 0.7363 - val_loss: 1.5042
Epoch 4/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 83s 99ms/step - accuracy: 0.7291 - loss: 1.5546 - val_accuracy: 0.7441 - val_loss: 1.4157
Epoch 5/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 81s 98ms/step - accuracy: 0.9015 - loss: 0.6283 - val_accuracy: 0.9819 - val_loss: 0.1559
Epoch 6/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 80s 95ms/step - accuracy: 0.9794 - loss: 0.1596 - val_accuracy: 0.9899 - val_loss: 0.0929
Epoch 7/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 73s 94ms/step - accuracy: 0.9869 - loss: 0.1044 - val_accuracy: 0.9836 - val_loss: 0.1233
Epoch 8/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 74s 96ms/step - accuracy: 0.9887 - loss: 0.0992

In [ ]:
spa_vocab = spa_vectorization.get_vocabulary()
spa_index_lookup = dict(zip(range(len(spa_vocab)), spa_vocab))
max_decoded_sentence_length = 20


def decode_sequence(input_sentence):
    tokenized_input_sentence = eng_vectorization([input_sentence])
    decoded_sentence = "[start]"
    for i in range(max_decoded_sentence_length):
        tokenized_target_sentence = spa_vectorization([decoded_sentence])[:, :-1]
        predictions = transformer([tokenized_input_sentence, tokenized_target_sentence])

        # ops.argmax(predictions[0, i, :]) is not a concrete value for jax here
        sampled_token_index = ops.convert_to_numpy(
            ops.argmax(predictions[0, i, :])
        ).item(0)
        sampled_token = spa_index_lookup[sampled_token_index]
        decoded_sentence += " " + sampled_token

        if sampled_token == "[end]":
            break
    return decoded_sentence


test_eng_texts = [pair[0] for pair in test_pairs]
for _ in range(10):
    input_sentence = random.choice(test_eng_texts)
    translated = decode_sequence(input_sentence)
    print(input_sentence)
    print(translated)
    print('=====================')

You can see that the architect paid scrupulous attention to detail.
[start] muy sé el [end]
It must have been terrible.
[start] decir una ellos iglesia                
They smiled.
[start] gustaba                   
I guess I'm not as smart as you.
[start] alrededor le al fue manzanas fue que             
Watch me.
[start] por                   
I'm pretty sure Tom has already made up his mind.
[start] dar ni no quiero bueno alguien tiempo las joven           
Why didn't you help Tom?
[start] como que mañana no                
Tom pulled the trigger.
[start] ajedrez [end]
We're here to say goodbye.
[start] todo de hizo nosotras                
I'll return to my country someday soon.
[start] decirle de los error tuviera aún              


In [ ]:
    test=decode_sequence("how are you ")
    print(test)

In [ ]:
# Fit the transformer model
history = transformer.fit(train_ds, epochs=epochs, validation_data=val_ds)


Epoch 1/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 74s 95ms/step - accuracy: 0.9992 - loss: 0.0151 - val_accuracy: 0.9803 - val_loss: 0.1343
Epoch 2/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 81s 94ms/step - accuracy: 0.9974 - loss: 0.0287 - val_accuracy: 0.9995 - val_loss: 0.0092
Epoch 3/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 73s 94ms/step - accuracy: 0.9997 - loss: 0.0064 - val_accuracy: 0.9993 - val_loss: 0.0115
Epoch 4/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 73s 95ms/step - accuracy: 0.9997 - loss: 0.0058 - val_accuracy: 0.9997 - val_loss: 0.0058
Epoch 5/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 81s 94ms/step - accuracy: 0.9997 - loss: 0.0060 - val_accuracy: 0.9998 - val_loss: 0.0046
Epoch 6/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 72s 93ms/step - accuracy: 0.9999 - loss: 0.0019 - val_accuracy: 0.9998 - val_loss: 0.0036
Epoch 7/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 83s 94ms/step - accuracy: 0.9999 - loss: 0.0015 - val_accuracy: 0.9998 - val_loss: 0.0034
Epoch 8/10
775/775 ━━━━━━━━━━━━━━━━━━━━ 73s 94ms/step - accuracy: 0.9999 - loss: 0.0012 - 

In [ ]:
test_eng_texts = [pair[0] for pair in test_pairs]
test_spa_texts = [pair[1] for pair in test_pairs]

for _ in range(10):
    idx = random.randint(0, len(test_eng_texts) - 1)
    input_sentence = test_eng_texts[idx]
    translated = test_spa_texts[idx]


    print(f"Input: {input_sentence}")
    print(f"Translated: {translated}")
    print("===============================")


Input: Do you hear me?
Translated: [start] ¿Me oyes? [end]
Input: I need to get to work.
Translated: [start] Necesito ir a trabajar. [end]
Input: The phone stopped working.
Translated: [start] El teléfono dejó de funcionar. [end]
Input: Tom came and sat down next to Mary.
Translated: [start] Tom vino y se sentó al lado de Mary. [end]
Input: Give me some time to think.
Translated: [start] Dame un poco de tiempo para pensar. [end]
Input: She must be ill in bed.
Translated: [start] Debe de estar en la cama enfermo. [end]
Input: Wisdom does not automatically come with age.
Translated: [start] La sabiduría no viene automáticamente con los años. [end]
Input: I was on the point of leaving home when a light rain started to fall.
Translated: [start] Estaba a punto de dejar la casa cuando empezó a caer una ligera lluvia. [end]
Input: Tom has been traveling all year.
Translated: [start] Tom ha estado viajando todo el año. [end]
Input: Tom was the best by far.
Translated: [start] Tom fue el mejor 